# Chapter 24 — MySQL from Tkinter: connection, CRUD, transactions, and configuration

**Book role:** Databases  
**Suggested book space:** about 11 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Connect with MySQL Connector/Python and/or SQLAlchemy.
- Keep credentials outside source code and set connection timeouts.
- Use parameterized CRUD and explicit transaction boundaries.
- Handle database failures without freezing the GUI.

## 1. Connection configuration and least privilege

Create a database account with only the permissions required by the application: SELECT for read-only reporting, and narrowly scoped INSERT/UPDATE/DELETE if the app edits data. Do not use a database administrator account in a desktop application. A desktop client’s embedded credentials can be extracted, so distributed applications that require strong security should consider a trusted service API rather than storing privileged secrets on every user machine.

Use environment variables for local development settings such as host, port, database, username, and password. Avoid printing a full connection URI in notebooks, logs, or error messages. Set connection and query timeouts where supported, validate input before opening a connection, and close cursors/connections reliably. A successful import of `mysql.connector` does not prove a MySQL server is running or reachable.

### Worked example 1: Load MySQL settings without embedding secrets

This is a local configuration pattern. Set actual credentials outside the notebook, and never print the password. For production, review secret storage and whether direct database access from client desktops is appropriate.

In [ ]:
import os

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "analytics_reader"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
    "database": os.getenv("MYSQL_DATABASE", "analytics"),
    "connection_timeout": 5,
}
if not MYSQL_CONFIG["password"]:
    print("MYSQL_PASSWORD is not configured; MySQL actions should remain disabled.")
print({key: value for key, value in MYSQL_CONFIG.items() if key != "password"})

## 2. Parameterized operations and transactions

MySQL Connector/Python uses `%s` placeholders for bound values. Values must be passed separately to `cursor.execute(sql, params)`; do not quote placeholders manually or use f-strings to insert values. For queries that return rows, fetch only what the user needs and convert records to a DataFrame with a known column list. For writes, inspect affected-row counts and commit only after the operation is complete.

Transactions matter when multiple updates form one business operation. If saving an order includes an order header and several line items, partial persistence is unacceptable. Begin a transaction, write the related records, commit if all steps succeed, and roll back on failure. In SQLAlchemy, transaction context managers simplify this pattern; use one approach consistently rather than mixing connection ownership casually.

### Worked example 2: Execute a parameterized query and close resources

The SQL string contains fixed identifiers and bound date values. The example deliberately does not invoke the server because credentials and schema are environment-specific.

In [ ]:
import os
import pandas as pd
import mysql.connector

# Run only after configuring environment variables and starting a permitted MySQL server.
def fetch_orders(start_date, end_date):
    connection = mysql.connector.connect(
        host=os.environ["MYSQL_HOST"],
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.environ["MYSQL_USER"],
        password=os.environ["MYSQL_PASSWORD"],
        database=os.environ["MYSQL_DATABASE"],
        connection_timeout=5,
    )
    try:
        cursor = connection.cursor(dictionary=True)
        try:
            cursor.execute(
                "SELECT order_id, order_date, region, revenue FROM orders "
                "WHERE order_date >= %s AND order_date < %s ORDER BY order_date, order_id",
                (start_date, end_date),
            )
            rows = cursor.fetchall()
            return pd.DataFrame(rows, columns=["order_id", "order_date", "region", "revenue"])
        finally:
            cursor.close()
    finally:
        connection.close()

# Example invocation intentionally omitted: it requires a configured real database.

## 3. Threading and resource lifetime

Never run an unpredictable network query directly inside a UI button callback. The connection attempt itself may block before any rows return. A worker can perform the database operation and send plain results/errors through a queue. The UI thread then enables/disables controls, updates status, and displays the returned DataFrame.

Do not casually share one live connection across multiple worker threads. Follow the driver’s thread-safety and pooling requirements. Create a connection per unit of work or use a configured connection pool. Close cursors, release connections, and ensure shutdown cancels or waits for outstanding operations according to a documented policy. A UI “Cancel” button should not claim cancellation unless the database operation can actually be interrupted.

### Worked example 3: Write multiple related rows transactionally

The helper expects a live connection and a schema matching the SQL. Its caller should handle the raised error, log useful context without private values, and surface a user-facing failure. Verify whether the driver manages autocommit as expected.

In [ ]:
def insert_order(connection, order_row, item_rows):
    """Insert a header and all lines as one transaction; schema is illustrative."""
    cursor = connection.cursor()
    try:
        cursor.execute(
            "INSERT INTO orders (order_id, order_date, customer_id) VALUES (%s, %s, %s)",
            (order_row["order_id"], order_row["order_date"], order_row["customer_id"]),
        )
        cursor.executemany(
            "INSERT INTO order_items (order_id, product_id, quantity, unit_price) "
            "VALUES (%s, %s, %s, %s)",
            [(order_row["order_id"], item["product_id"], item["quantity"], item["unit_price"]) for item in item_rows],
        )
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()

## Guided lab: Lab: safe MySQL connector

Create a MySQL adapter with `test_connection()`, `fetch_sales(criteria)`, and a write method only if needed. Use environment variables, a least-privilege user, parameterized SQL, explicit transaction handling, and deterministic resource cleanup. Show a GUI status for connecting/loading/failed/loaded. Keep all live operations behind buttons and do not make a database connection during notebook import.

## Review questions and practice

1. Why should a desktop application not use an administrator database account?
2. How does Connector/Python bind query values?
3. Why should a multi-table write use a transaction?
4. Why is a separate worker appropriate for network I/O?
5. Why should the UI avoid exposing raw connection exceptions without context?

## Further reading

- MySQL Connector/Python: https://dev.mysql.com/doc/connector-python/en/
- SQLAlchemy connections: https://docs.sqlalchemy.org/en/20/core/connections.html
- MySQL transactions: https://dev.mysql.com/doc/refman/8.4/en/commit.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.